# NB01: Data Collection

The research aim for this project is to identify whether the winner of a professional tennis match is predicted by hitting more winners (aggression) or by committing fewer unforced errors (consistency). 

This notebook collects data from the `get_tournaments` and `get_fixtures` endpoints from API-Tennis.

## Setup

In [1]:
import os
import json
import requests
from dotenv import load_dotenv
import pandas as pd

Loading the API key (replace the placeholder in the .env with your own key):

In [ ]:
load_dotenv()

API_KEY = os.getenv("API_KEY")

## Context

### Tournaments

For the project, the tournaments chosen were the 3 Grand Slams completed thus far this year (as of July 2026). Only the men's singles draws were considered, as restricting to one tour only removes the need to control for differing tournament formats and player playstyles. 

Grand Slams were chosen deliberately for three reasons:

1. **Larger sample size**: Draws for Grand Slams contain more players than lower tier ATP tour events and thus more data points to investigate.
2. **Data completeness**: `statistics` are recorded much more consistently and reliably for high-profile tour events.
3. **Surface variety**: The three slams played so far (Australian Open, Roland-Garros, and Wimbledon) coincide with the three different playing surfaces (Hard, clay, grass respectively) which allows for further analysis on whether insights change depending on the surface the match is played on.


First we have to obtain the tournament keys for the tournaments of interest:

In [ ]:
url = "https://api.api-tennis.com/tennis/"
params = {
    "method": "get_tournaments",
    "APIkey": API_KEY
}

response = requests.get(url, params=params)
tournaments = response.json()["result"]

print(f"Status code: {response.status_code}")
print(f"Number of tournaments: {len(tournaments)}")

# restrict to the Slams that we want
slam_names = ["Australian Open", "French Open", "Wimbledon"]

# filter to ATP singles only, excluding doubles, WTA etc that would also come up as they have the same tournament name
slam_tournaments = [t for t in tournaments if (t["tournament_name"] in slam_names) & ("Atp Singles" in t["event_type_type"])]

Status code: 200
Number of tournaments: 10155


In [4]:
slam_tournaments

[{'tournament_key': 1236,
  'tournament_name': 'Australian Open',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Hard'},
 {'tournament_key': 2155,
  'tournament_name': 'French Open',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Clay'},
 {'tournament_key': 2053,
  'tournament_name': 'Wimbledon',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Grass'}]

Now that we have the `tournament_key` for each Slam, we can work on obtaining the `statistics` field from the `get_fixtures` endpoint.

### Variables of interest

Identifying fields (player keys, match winners, tournament name) and the full statistics list are kept from each match, from which `NB02-Data-Transformation.ipynb` will extract more required variables more precisely.


In [ ]:
START_DATE = "2026-01-01"
STOP_DATE = "2026-07-31"

# keep only the fields needed for analysis
to_keep = ["first_player_key", "second_player_key", "event_winner", "tournament_name", "statistics"]

datalist = []

# loop over once per slam
for details in slam_tournaments:
    params = {
        "method": "get_fixtures",
        "APIkey": API_KEY,
        "date_start": START_DATE,
        "date_stop": STOP_DATE,
        "event_type_key": details["event_type_key"],
        "tournament_key": details["tournament_key"],
    }   

    response = requests.get(url, params=params)
    if reponse.status_code != 200:
        print(f"Warning, {details["tournament_name"]} returned an error with status code {response.status_code}, skipping")
        continue


    placeholder = response.json()

    full_results = placeholder["result"]
    for fixture in full_results:
        # keep only the necessary fields to not bloat the raw data file
        fixture_edited = {key: val for key,val in fixture.items() if (key in to_keep)}
        datalist.append(fixture_edited)

print(len(datalist))

728


Each Grand Slam has 239 matches (including qualifying), which should mean that our data should contain 717 values, but our dataset has 11 extra recordings. From a quick inspection of the data as a dataframe:

In [ ]:
df = pd.DataFrame(datalist)

df[df['statistics'].str.len() == False] # empty list returns false

,first_player_key,second_player_key,event_winner,tournament_name,statistics
10,1205,2985,NaN,ATP Australian Open,[]
25,1764,1896,NaN,ATP Australian Open,[]
50,1899,439,NaN,ATP Australian Open,[]
119,2844,1106,NaN,ATP Australian Open,[]
165,1067,824,NaN,ATP Australian Open,[]
235,902,1905,Second Player,Australian Open,[]
295,39675,362,NaN,French Open,[]
332,5979,1748,Second Player,French Open,[]
358,2168,1759,NaN,French Open,[]
360,1066,8130,NaN,French Open,[]


There are 16 extra findings that do not have any statistics tied to them. These refer to pre-match withdrawals (as opposed to mid-match retirements, as no statistics have been recorded). If a player withdraws from a qualifying round or the first round of the tournament, they are replaced and the match is played with an alternate. If the withdrawal occurs in a latter round, the match is forfeited. 

With the `event_winner` tab, we can identify whether a player withdrew from the early or latter stages of the tournament. As we can see above, 5 withdrawals were from the 2nd round onwards, and that leaves 11 qualifying/first round withdrawals, which coincides with the 11 extra match results. All 16 of these matches will be removed in `NB02-Data-Transformation.ipynb`.

Now to output to json:

In [7]:
file_path = '../data/raw/dump.json'
with open(file_path, "w") as f:
    json.dump(datalist, f, indent=4)